### Name: Ashish Mulani
### PRN: 260240128007
------------------------------
### Name: Pranita Lad
### PRN: 260240128028

In [15]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.ensemble import BaggingRegressor, StackingRegressor
from catboost import CatBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, GridSearchCV
from tqdm import tqdm 
import os
os.chdir("D:/download chrome/RoadAccident_Kaggle")

In [4]:
train = pd.read_csv('train.csv',index_col=0)
test = pd.read_csv('test.csv',index_col=0)
train

,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
id,,,,,,,,,,,,,
0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,False,True,1,0.13
1,urban,4,0.99,35,daylight,clear,True,False,evening,True,True,0,0.35
2,rural,4,0.63,70,dim,clear,False,True,morning,True,False,2,0.30
3,highway,4,0.07,35,dim,rainy,True,True,morning,False,False,1,0.21
4,rural,1,0.58,60,daylight,foggy,False,False,evening,True,False,1,0.56
...,...,...,...,...,...,...,...,...,...,...,...,...,...
517749,highway,4,0.10,70,daylight,foggy,True,True,afternoon,False,False,2,0.32
517750,rural,4,0.47,35,daylight,rainy,True,True,morning,False,False,1,0.26
517751,urban,4,0.62,25,daylight,foggy,False,False,afternoon,False,True,0,0.19


In [7]:
ohe = OneHotEncoder(sparse_output=False, drop='first').set_output(transform='pandas')
transf = ColumnTransformer(transformers=[('OHE', 
       ohe, make_column_selector(dtype_include=object) )] ,
    remainder='passthrough',verbose_feature_names_out=False).set_output(transform='pandas')

In [16]:
X, y = train.drop('accident_risk', axis=1), train['accident_risk']
kfold = KFold(n_splits=5, shuffle=True, random_state=26)

In [ ]:
lr = LinearRegression()
el = ElasticNet()
knn = KNeighborsRegressor()
dtr = DecisionTreeRegressor(random_state=26)
rf = RandomForestRegressor(n_estimators=50, random_state=26)

## Bagging

In [ ]:

bagg = BaggingRegressor(random_state=26)

pipe_bagg = Pipeline([('OHE', transf), ('BAGG', bagg)])

params = {
    'BAGG__estimator': [lr, el, knn, dtr],
    'BAGG__n_estimators': [10, 15, 50, 100]}

gcv_bagg = GridSearchCV(pipe_bagg, param_grid=params, cv=kfold, scoring='r2', n_jobs=-1)
gcv_bagg.fit(X, y)

df_results = pd.DataFrame(gcv_bagg.cv_results_)
print(df_results[['param_BAGG__estimator', 'param_BAGG__n_estimators', 'mean_test_score']])

In [ ]:
# Show full GridSearch results for Bagging
df_bagg = pd.DataFrame(gcv_bagg.cv_results_)
df_bagg['estimator_name'] = df_bagg['param_BAGG__estimator'].apply(
    lambda e: type(e).__name__
)
display(
    df_bagg[['estimator_name', 'param_BAGG__n_estimators', 'mean_test_score', 'std_test_score']]
    .sort_values('mean_test_score', ascending=False)
    .reset_index(drop=True)
)

In [ ]:
# --- BAGGING Submission ---
y_pred_bagg = gcv_bagg.best_estimator_.predict(test)
y_pred_bagg = np.clip(y_pred_bagg, 0, None)   # accident_risk cannot be negative

sub_bagg = sample_sub.copy()
sub_bagg['accident_risk'] = y_pred_bagg
sub_bagg.to_csv('submission_bagging.csv', index=False)
print("submission_bagging.csv saved ✓")
sub_bagg.head()

## Cat Boost

In [ ]:
n_est = [10, 25, 50, 75, 100]
rate = np.linspace(0.0001, 0.9, 30)
depth = [2, 3, 4, 5, 6]
scores = []
for n in tqdm(n_est):
    for r in rate:
        for d in depth:
            gbm = CatBoostRegressor(random_state=26, n_estimators=n,
                                            learning_rate=r, max_depth=d, verbose=0)
            gbm.fit(X_train, y_train)
            y_pred = gbm.predict(X_test)
            scores.append([n, r, d, r2_score(y_test, y_pred)])
df_scores = pd.DataFrame(scores, columns=['n_est','rate','depth','score'])
df_scores.sort_values('score', ascending=False)

## Stacking

In [ ]:
stack = StackingRegressor(estimators=[('LR', lr), ('EL', el), ('KNN', knn), ('DTR', dtr),('RF',rf)],
                            passthrough=True, cv=5, n_jobs=-1)

pipe_stack = Pipeline([('OHE', transf), ('STACK', stack)])

params_stack = {
    'STACK__final_estimator': [Ridge(), XGBRegressor(random_state=26)]
}

gcv_stack = GridSearchCV(pipe_stack, param_grid=params_stack, cv=kfold, scoring='r2', n_jobs=-1)
gcv_stack.fit(X, y)